# Tutorial 5: Unsupervised Learning — PCA and Clustering

**Big Data in Finance** | Dr Daniele Bianchi  
**Duration:** 1 hour (+ 15 min bonus)

---

## Learning Objectives

By the end of this tutorial, you will be able to:

1. Apply Principal Component Analysis (PCA) to reduce dimensionality
2. Interpret PCA results: loadings, variance explained, scree plots
3. Use K-means and hierarchical clustering to identify market regimes
4. Validate clusters using silhouette analysis and the elbow method
5. Use AI assistants to debug code and improve efficiency

---

## How This Tutorial Works

- **Run the code cells** in order (`Shift + Enter`)
- 🔧 **Exercise**: Short tasks to test your understanding
- 🤖 **AI Prompt**: Practice using ChatGPT/Claude for debugging, efficiency, and interpretation
- 💡 **Tip**: Helpful hints

> 💡 **Note (scope):** This tutorial uses a deliberately simplified setup for a one-hour session, so its results are illustrative and need not match the lecture figures exactly.

---

## Part 1: Setup and Data Preparation

We continue using the Goyal-Welch-Zafirov (GWZ) dataset of macroeconomic predictors.

In [ ]:
# =============================================================================
# Import libraries
# =============================================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Unsupervised learning
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, silhouette_samples

# Suppress warnings for cleaner output
import warnings
warnings.filterwarnings('ignore')

# Plot style
plt.style.use('seaborn-v0_8-whitegrid')

print("Libraries loaded successfully!")

In [ ]:
# =============================================================================
# Load the GWZ dataset
# =============================================================================

df = pd.read_csv('Data_GWZ.csv')

# Parse date
df['DATE'] = pd.to_datetime(df['DATE'], format='%Y%m')
df = df.set_index('DATE')

print(f"Dataset shape: {df.shape}")
print(f"Date range: {df.index.min()} to {df.index.max()}")
df.head()

In [ ]:
# =============================================================================
# Separate target and features
# =============================================================================

# Target: Market excess return
y = df['MktRf']

# Features: All predictor variables (exclude MktRf)
X = df.drop(columns=['MktRf'])

# Drop rows with missing values
mask = X.notna().all(axis=1) & y.notna()
X = X[mask]
y = y[mask]

print(f"Features shape: {X.shape}")
print(f"\nFeatures: {list(X.columns)}")

### 1.1 Why Standardization Matters for PCA

PCA finds directions of **maximum variance**. If features are on different scales, those with larger variance will dominate the principal components.

💡 **Tip:** Always standardize before PCA unless you have a specific reason not to.

In [ ]:
# =============================================================================
# Standardize features (mean=0, std=1)
# =============================================================================

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Convert back to DataFrame for easier inspection
X_scaled_df = pd.DataFrame(X_scaled, columns=X.columns, index=X.index)

print("Before standardization:")
print(X.describe().loc[['mean', 'std']].round(4))
print("\nAfter standardization:")
print(X_scaled_df.describe().loc[['mean', 'std']].round(4))

---

## Part 2: Principal Component Analysis (PCA)

### 2.1 Fitting PCA

PCA finds new variables (principal components) that are:
1. **Linear combinations** of the original features
2. **Uncorrelated** with each other
3. Ordered by **variance explained** (PC1 explains the most)

In [ ]:
# =============================================================================
# Fit PCA - keep all components initially
# =============================================================================

pca = PCA()  # No n_components specified = keep all
pca.fit(X_scaled)

# Variance explained by each component
var_explained = pca.explained_variance_ratio_
cumulative_var = np.cumsum(var_explained)

print("Variance explained by each PC:")
for i in range(min(10, len(var_explained))):
    print(f"  PC{i+1}: {var_explained[i]*100:.1f}% (cumulative: {cumulative_var[i]*100:.1f}%)")

### 2.2 The Scree Plot

A scree plot shows variance explained by each component. Look for the **"elbow"** — where adding more components gives diminishing returns.

In [ ]:
# =============================================================================
# Scree plot
# =============================================================================

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Individual variance
axes[0].bar(range(1, len(var_explained)+1), var_explained * 100, 
            color='steelblue', edgecolor='white')
axes[0].set_xlabel('Principal Component')
axes[0].set_ylabel('Variance Explained (%)')
axes[0].set_title('Scree Plot: Individual Variance')
axes[0].set_xticks(range(1, len(var_explained)+1, 2))

# Cumulative variance
axes[1].plot(range(1, len(cumulative_var)+1), cumulative_var * 100, 
             'o-', color='steelblue', linewidth=2, markersize=6)
axes[1].axhline(y=80, color='red', linestyle='--', label='80% threshold')
axes[1].axhline(y=90, color='orange', linestyle='--', label='90% threshold')
axes[1].set_xlabel('Number of Components')
axes[1].set_ylabel('Cumulative Variance Explained (%)')
axes[1].set_title('Cumulative Variance Explained')
axes[1].legend()
axes[1].set_xticks(range(1, len(cumulative_var)+1, 2))

plt.tight_layout()
plt.show()

# How many components for 80% and 90%?
n_80 = np.argmax(cumulative_var >= 0.80) + 1
n_90 = np.argmax(cumulative_var >= 0.90) + 1
print(f"\nComponents needed for 80% variance: {n_80}")
print(f"Components needed for 90% variance: {n_90}")

### 🤖 AI Prompt Exercise 2.1: Debug and Improve Code

The code below has a bug. Use an AI assistant to identify and fix it.

**Copy this prompt:**
```
I'm trying to get the number of PCA components needed to explain at least 85% of variance. My code returns the wrong answer:

cumulative_var = np.cumsum(pca.explained_variance_ratio_)
n_components = np.where(cumulative_var > 0.85)[0]
print(f"Need {n_components} components for 85% variance")

What's wrong and how do I fix it?
```

In [ ]:
# Try the buggy code, then fix it based on AI feedback
cumulative_var = np.cumsum(pca.explained_variance_ratio_)
n_components = np.where(cumulative_var > 0.85)[0]
print(f"Need {n_components} components for 85% variance")  # This prints an array, not a number!

# Your fixed code here:


### 2.3 Interpreting PCA Loadings

**Loadings** tell us how each original feature contributes to each principal component.

In [ ]:
# =============================================================================
# PCA Loadings (first 5 components)
# =============================================================================

n_components_to_show = 5
loadings = pd.DataFrame(
    pca.components_[:n_components_to_show].T,
    columns=[f'PC{i+1}' for i in range(n_components_to_show)],
    index=X.columns
)

print("PCA Loadings (first 5 components):")
print(loadings.round(3))

In [ ]:
# =============================================================================
# Visualize loadings for PC1 and PC2
# =============================================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# PC1 loadings
loadings['PC1'].sort_values().plot(kind='barh', ax=axes[0], color='steelblue')
axes[0].set_title('PC1 Loadings (explains {:.1f}% variance)'.format(var_explained[0]*100))
axes[0].set_xlabel('Loading')
axes[0].axvline(x=0, color='black', linewidth=0.5)

# PC2 loadings
loadings['PC2'].sort_values().plot(kind='barh', ax=axes[1], color='darkorange')
axes[1].set_title('PC2 Loadings (explains {:.1f}% variance)'.format(var_explained[1]*100))
axes[1].set_xlabel('Loading')
axes[1].axvline(x=0, color='black', linewidth=0.5)

plt.tight_layout()
plt.show()

### 🔧 Exercise 2.2: Interpret the Components

Look at the loadings above and answer:
1. Which variables load most heavily on PC1? What might PC1 represent economically?
2. Which variables load most heavily on PC2? What might PC2 represent?

*Write your interpretation below:*

*Your interpretation:*



### 2.4 Transforming Data to PC Space

In [ ]:
# =============================================================================
# Transform data to principal component scores
# =============================================================================

# Keep first 5 components
pca_5 = PCA(n_components=5)
pc_scores = pca_5.fit_transform(X_scaled)

# Create DataFrame with PC scores
pc_df = pd.DataFrame(
    pc_scores,
    columns=[f'PC{i+1}' for i in range(5)],
    index=X.index
)

print("Principal Component Scores (first 5 rows):")
print(pc_df.head())

print(f"\nOriginal features: {X.shape[1]} variables")
print(f"After PCA: {pc_df.shape[1]} components")
print(f"Variance retained: {pca_5.explained_variance_ratio_.sum()*100:.1f}%")

In [ ]:
# =============================================================================
# Plot PC1 over time
# =============================================================================

fig, ax = plt.subplots(figsize=(12, 4))

ax.plot(pc_df.index, pc_df['PC1'], color='steelblue', linewidth=0.8)
ax.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
ax.set_title('PC1 Over Time')
ax.set_xlabel('Date')
ax.set_ylabel('PC1 Score')

# Shade recessions (approximate NBER dates)
recessions = [
    ('1973-11-01', '1975-03-01'),
    ('1980-01-01', '1980-07-01'),
    ('1981-07-01', '1982-11-01'),
    ('1990-07-01', '1991-03-01'),
    ('2001-03-01', '2001-11-01'),
    ('2007-12-01', '2009-06-01'),
    ('2020-02-01', '2020-04-01')
]
for start, end in recessions:
    ax.axvspan(pd.to_datetime(start), pd.to_datetime(end), 
               alpha=0.2, color='gray', label='_nolegend_')

plt.tight_layout()
plt.show()

---

## Part 3: K-Means Clustering

Now we'll use clustering to identify **market regimes** — periods when market conditions are similar.

### 3.1 The Elbow Method for Choosing K

In [ ]:
# =============================================================================
# Elbow method: try different values of K
# =============================================================================

# Use PC scores for clustering (reduces noise)
X_cluster = pc_df.values

# Try K from 2 to 10
K_range = range(2, 11)
inertias = []
silhouettes = []

for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_cluster)
    inertias.append(kmeans.inertia_)
    silhouettes.append(silhouette_score(X_cluster, kmeans.labels_))

# Plot
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Elbow plot (inertia)
axes[0].plot(K_range, inertias, 'o-', color='steelblue', linewidth=2, markersize=8)
axes[0].set_xlabel('Number of Clusters (K)')
axes[0].set_ylabel('Inertia (Within-cluster SS)')
axes[0].set_title('Elbow Method')

# Silhouette plot
axes[1].plot(K_range, silhouettes, 'o-', color='darkorange', linewidth=2, markersize=8)
axes[1].set_xlabel('Number of Clusters (K)')
axes[1].set_ylabel('Silhouette Score')
axes[1].set_title('Silhouette Analysis')

plt.tight_layout()
plt.show()

print(f"Best K by silhouette score: {K_range[np.argmax(silhouettes)]}")

### 🤖 AI Prompt Exercise 3.1: Make the Code More Efficient

The loop above runs K-means 9 times sequentially. Ask an AI to make it faster.

**Copy this prompt:**
```
I have this Python code that runs K-means clustering in a loop:

K_range = range(2, 11)
inertias = []
silhouettes = []

for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_cluster)
    inertias.append(kmeans.inertia_)
    silhouettes.append(silhouette_score(X_cluster, kmeans.labels_))

How can I make this more efficient? Consider:
1. Parallel processing options
2. More Pythonic alternatives to the loop
3. Any sklearn built-in functions that could help
```

*Write the improved code below:*

In [ ]:
# Your improved code here:


### 3.2 Fit K-Means with Chosen K

In [ ]:
# =============================================================================
# Fit K-means with K=3 (common choice for market regimes)
# =============================================================================

K = 3
kmeans = KMeans(n_clusters=K, random_state=42, n_init=10)
cluster_labels = kmeans.fit_predict(X_cluster)

# Add cluster labels to our data
pc_df['Cluster'] = cluster_labels

print(f"Cluster distribution:")
print(pc_df['Cluster'].value_counts().sort_index())
print(f"\nSilhouette score: {silhouette_score(X_cluster, cluster_labels):.3f}")

In [ ]:
# =============================================================================
# Visualize clusters in PC1-PC2 space
# =============================================================================

fig, ax = plt.subplots(figsize=(10, 6))

colors = ['steelblue', 'darkorange', 'green']
for cluster in range(K):
    mask = pc_df['Cluster'] == cluster
    ax.scatter(pc_df.loc[mask, 'PC1'], pc_df.loc[mask, 'PC2'],
               c=colors[cluster], label=f'Cluster {cluster}', alpha=0.6, s=30)

# Plot centroids
centroids = kmeans.cluster_centers_
ax.scatter(centroids[:, 0], centroids[:, 1], c='red', marker='X', 
           s=200, edgecolors='black', linewidths=2, label='Centroids')

ax.set_xlabel('PC1')
ax.set_ylabel('PC2')
ax.set_title('K-Means Clustering (K=3) in PC Space')
ax.legend()
plt.tight_layout()
plt.show()

### 3.3 Interpreting Clusters as Market Regimes

In [ ]:
# =============================================================================
# Analyze clusters: what are the average returns in each regime?
# =============================================================================

# Add returns to the PC dataframe
pc_df['MktRf'] = y.values

# Summary statistics by cluster
cluster_stats = pc_df.groupby('Cluster')['MktRf'].agg(['mean', 'std', 'count'])
cluster_stats['mean'] = cluster_stats['mean'] * 100  # Convert to percentage
cluster_stats['std'] = cluster_stats['std'] * 100
cluster_stats['sharpe'] = cluster_stats['mean'] / cluster_stats['std'] * np.sqrt(12)  # Annualized
cluster_stats.columns = ['Mean Return (%)', 'Std Dev (%)', 'N Months', 'Sharpe (Ann.)']

print("Market Return Statistics by Cluster:")
print(cluster_stats.round(2))

In [ ]:
# =============================================================================
# Plot cluster regimes over time
# =============================================================================

fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)

# Top: Market returns
axes[0].plot(pc_df.index, pc_df['MktRf']*100, color='black', linewidth=0.5, alpha=0.7)
axes[0].axhline(y=0, color='gray', linestyle='-', linewidth=0.5)
axes[0].set_ylabel('Market Return (%)')
axes[0].set_title('Market Returns and Cluster Regimes')

# Bottom: Cluster assignments
for cluster in range(K):
    mask = pc_df['Cluster'] == cluster
    axes[1].fill_between(pc_df.index, 0, 1, where=mask, 
                         alpha=0.7, color=colors[cluster], label=f'Cluster {cluster}')

axes[1].set_ylabel('Regime')
axes[1].set_xlabel('Date')
axes[1].legend(loc='upper right')
axes[1].set_yticks([])

plt.tight_layout()
plt.show()

### 🔧 Exercise 3.2: Name the Regimes

Based on the return statistics and the time plot above:
1. Which cluster appears to be the "crisis" or "bear" regime?
2. Which cluster is the "normal" or "expansion" regime?
3. What might the third cluster represent?

*Write your interpretation:*

*Your interpretation:*



---

## Key Takeaways

1. **PCA reduces dimensionality** while preserving most variance
   - Always standardize features first
   - Use scree plot to choose number of components
   - Loadings help interpret what each PC represents

2. **Clustering groups similar observations**
   - K-means: fast, requires specifying K
   - Validate with silhouette score and elbow method

3. **In finance, clusters can represent market regimes**
   - Different return characteristics in each regime
   - Can be used for conditional strategies

4. **PCA + Clustering is a common workflow**
   - First reduce dimension with PCA
   - Then cluster in the lower-dimensional space

---

## Next Week

**Topic:** Model Interpretability (SHAP, Feature Importance, PDP)

**Reading:** Molnar (2022), *Interpretable Machine Learning*, Chapters 5-9

---

## 🔧 Bonus Section (If Time Permits)

### Bonus 1: Clustering on Original Features vs. PC Scores

In [ ]:
# Compare clustering on raw standardized features vs PC scores
# Which gives better silhouette score?

# Cluster on original standardized features
kmeans_raw = KMeans(n_clusters=3, random_state=42, n_init=10)
labels_raw = kmeans_raw.fit_predict(X_scaled)

# Cluster on PC scores (already done)
labels_pc = cluster_labels

print("Silhouette Scores:")
print(f"  Clustering on raw features: {silhouette_score(X_scaled, labels_raw):.3f}")
print(f"  Clustering on PC scores:    {silhouette_score(X_cluster, labels_pc):.3f}")

### Bonus 2: Regime-Switching Strategy Backtest

In [ ]:
# Simple regime-switching strategy:
# - Invest in market during "good" regimes
# - Go to cash during "bad" regimes

# Identify worst-performing cluster
cluster_returns = pc_df.groupby('Cluster')['MktRf'].mean()
bad_cluster = cluster_returns.idxmin()

print(f"Worst cluster (by average return): {bad_cluster}")
print(f"Average return in bad cluster: {cluster_returns[bad_cluster]*100:.2f}%")

# Strategy: avoid market during bad cluster
pc_df['Strategy'] = np.where(pc_df['Cluster'] == bad_cluster, 0, pc_df['MktRf'])

# Compare cumulative returns (Note: this has look-ahead bias!)
pc_df['Cum_Market'] = (1 + pc_df['MktRf']).cumprod()
pc_df['Cum_Strategy'] = (1 + pc_df['Strategy']).cumprod()

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(pc_df.index, pc_df['Cum_Market'], label='Buy & Hold', linewidth=1.5)
ax.plot(pc_df.index, pc_df['Cum_Strategy'], label='Regime-Switching', linewidth=1.5)
ax.set_ylabel('Cumulative Return')
ax.set_title('Regime-Switching Strategy (⚠️ Has Look-Ahead Bias!)')
ax.legend()
ax.set_yscale('log')
plt.tight_layout()
plt.show()

print("\n⚠️ WARNING: This backtest uses future information (look-ahead bias).")
print("In practice, you would need to predict cluster membership out-of-sample.")

### 🤖 Bonus AI Exercise: Fix the Look-Ahead Bias

Ask an AI:
```
The regime-switching strategy above has look-ahead bias because we're using cluster assignments from data that includes future observations.

How would I modify this to be a valid out-of-sample backtest? Specifically:
1. How should I fit PCA and K-means to avoid look-ahead bias?
2. Should I use expanding or rolling windows?
3. What's the general principle for avoiding look-ahead bias in unsupervised learning?
```

In [ ]:
# Your implementation of a proper out-of-sample backtest (optional):
